# Week 1

## Downloaded and Manipulated Fits Files
We downloaded two fits files with 30 images each, then created a new difference fits file.

In [ ]:
import numpy as np

# Set up matplotlib
import matplotlib.pyplot as plt

from astropy.io import fits

# Extracting data from the light image file
light_image = fits.open(r"C:\Users\brie_\OneDrive - University of Arizona\Documents\2026 Summer Research LBT\Downloaded Files\Week 1\Linearity Images-20260518T043649Z-3-001\Linearity Images\lm_260429_000152.fits")
#light_image.info()
light_img_data = light_image[0].data
print(type(light_img_data)) # Show the Python type for image_data
print(light_img_data.shape) # Show the number of pixels per side in the 2-D image
light_image.close()

#Extracting data from the dark image file
dark_image = fits.open(r"C:\Users\brie_\OneDrive - University of Arizona\Documents\2026 Summer Research LBT\Downloaded Files\Week 1\Linearity Images-20260518T043649Z-3-001\Linearity Images\lm_260429_000162.fits")
dark_img_data = dark_image[0].data
dark_image.close()

#Plotting one frame from each set
plt.figure()
plt.imshow(light_img_data[1], cmap="gray")
plt.colorbar()

plt.figure()
plt.imshow(dark_img_data[1], cmap="gray")
plt.colorbar()

# plotting all frames from the light set together
fig1, axes1 = plt.subplots(6, 5, figsize = (18, 15))

axes1 = axes1.ravel()  # flatten the grid for easy indexing

for i in range(30):
    axes1[i].imshow(light_img_data[i], cmap="gray", origin="lower")
    axes1[i].set_title(f"Frame {i}", fontsize=10)
    axes1[i].axis("off")

print("fig1 is:", type(fig1))
fig1.suptitle("All Light Images", fontsize=30)
plt.tight_layout()
plt.show()

# plotting all frames from the dark set together
fig2, axes2 = plt.subplots(6, 5, figsize = (18, 15))
axes2 = axes2.ravel()  # flatten the grid for easy indexing

for i in range(30):
    axes2[i].imshow(dark_img_data[i], cmap="gray", origin="lower")
    axes2[i].set_title(f"Frame {i}", fontsize=10)
    axes2[i].axis("off")

fig2.suptitle("All Dark Images", fontsize=30)
plt.tight_layout()
plt.show()

# making a light - dark fits plot
net_light_data = light_img_data - dark_img_data

fig3, axes3 = plt.subplots(6, 5, figsize = (18, 15))
axes3 = axes3.ravel()  # flatten the grid for easy indexing

for i in range(30):
    axes3[i].imshow(net_light_data[i], cmap="gray", origin="lower")
    axes3[i].set_title(f"Frame {i}", fontsize=10)
    axes3[i].axis("off")

fig3.suptitle("Net Light Images", fontsize=30)
plt.tight_layout()
plt.show()

# making new net light FITS file
new_header = light_image[0].header.copy()
new_header['HISTORY'] = 'subtracted light image Im_260429_000152 from dark image Im_260429_000162'
# Write the FITS file
hdu = fits.PrimaryHDU(net_light_data, header=new_header)
hdu.writeto("net_light_images_week_1.fits", overwrite=True)

## Figures Created
![All light reads](Week_1/light_images_grid_wk1.png)  
![All dark reads](Week_1/dark_images_grid_wk1.png)  
![All difference reads](Week_1/net_images_grid_wk1.png)

### Notes
-this week I mostly worked to figure out the ds9 app and what fits files were  
-I wanted to see how the detector reads changed over time so I plotted all frames in one figure for the light, dark, and net images  
-I am still getting used to working with matplotlib so my code is quite messy here and I hope to improve my flow as I continue working  

# Week 2

## Downloaded Fits Files From LBTI Archive
I did this by selecting the files I wanted to download, then creating a .txt file of the urls to download in the interface then downloaded that to my computer. <br><br>
Next I went into my terminal and went into the folder I wanted my downloaded files to be in. Getting into the file in my terminal is done using basic bash:  
(in terminal) cd "path_to_desired_folder"<br><br>
Once in the folder I wanted to download to, I ran the following command in my terminal (this is for windows only but I'm sure a similar command can be looked up for mac, linux, etc.):  
(in terminal)

```powershell
foreach ($url in Get-Content "path_to_.txt_file") {
    Invoke-WebRequest -Uri $url -OutFile (Split-Path $url -Leaf)
}
```

This method allowed me to download files in the background while working on other tasks on my computer.  
It did require my laptop to be on, so if I closed it or it went to sleep the downloading would stop mid .txt file and whatever .fits file it was downloading would be corrupted. This was solved by changing the settings of my laptop to not go to sleep while charging.<br><br>
If a .fits download was stopped halfway through, I just deleted that downloaded file, took the completed downloaded files out of the .txt file, and reran the above code.

## Averaged Multiple Fits Files and Made Average Difference Files For Different Modes
Using the log, I organized downloaded files based on mode (fast, medium, slow) and light vs dark files. I opened these files then found the mean of the light and dark images separately for each mode, then created an average difference file for each mode.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob


def create_file_list(file_paths):
    file_list = glob.glob(file_paths)
    return file_list


# getting an average file for a given list of files, carrying over header
# filelist is a list of the full fits files, filetype is light or dark files
def average_files(filelist, filetype):

    #making a list of the data from each fits file
    file_concat = [fits.getdata(image) for image in filelist]

    #finding the mean (averaging the files)
    #.astype(np.float32) shrinks the acuracy of the estimation, since the mean auto
    #defaults to float64, so this will make the file smaller
    final_file_data = np.mean(file_concat, axis=0).astype(np.float32)

    #carry over first image header and add information on averaging
    original_header = fits.getheader(filelist[0])
    new_header = original_header.copy()
    # will return something of the style: averaged 10 light files
    new_header['HISTORY'] = f"averaged {len(filelist)} {filetype} files"
    # gives numeber of averaged files
    new_header['NFILES'] = len(filelist)
    new_header['METHOD'] = 'MEAN'

    return final_file_data, new_header

# making a net light file, and adjusting the header to reflect this
def calculate_difference(lightdata, darkdata, header):
    # calculate difference in light and dark files
    net_data = lightdata - darkdata

    #adjust header for new net_data file
    new_header = header.copy()
    new_header['HISTORY'] = f"difference between averaged light and dark file data"
    return net_data, new_header

#creating and saving a new fits file to my computer
def create_file(filename, filedata, header):
    hdu = fits.PrimaryHDU(filedata, header=header)
    hdu.writeto(filename, overwrite=True)

if __name__ == "__main__":
    # I kept the basic structure and only changed the folder when creating 
    # the dark and light file list, with each folder assigned to fast, med, or slow
    light_files = create_file_list(r"C:\Users\brie_\OneDrive - University of Arizona\Documents\2026 Summer Research LBT\Downloaded Files\Week 2\Week 2 Fits\Slow Light\*.fits.gz")
    avg_light, light_header = average_files(light_files, "light")
    dark_files = create_file_list(r"C:\Users\brie_\OneDrive - University of Arizona\Documents\2026 Summer Research LBT\Downloaded Files\Week 2\Week 2 Fits\Slow Dark\*.fits.gz")
    avg_dark, dark_header = average_files(dark_files, "dark")
    net_data, net_header = calculate_difference(avg_light, avg_dark, light_header)
    create_file("avg_slow_light_image_week2.fits", avg_light, light_header)
    create_file("avg_slow_dark_image_week2.fits", avg_dark, dark_header)
    create_file("net_slow_image_week2.fits", net_data, net_header)

## Plotting The Median Pixel Value for a Box In Each Frame
In total an average dark, light, and difference file was created for each of the three modes.  
For each mode, I selected a box of pixels in each image to average using the median, then plotted the median pixel value in the box as a function of time as each frame moved forward in time. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

# I chose to analyze a box from 970<x<1800 and 1070<y<1670 for all images
# In lab we discussed that this is a good choice since we want the box to 
# be as central as possible

# for a given fits file, cut down to the box, find the median of pixel values in this 
# box for each image, and add to the plot
# filename is the fits file, setting and color are both strings where setting describes
# the LBTI setting, file describes the fits file type, and color is the desired
# color of the plotted points for that file
def plot_avg(filename, setting, filetype, color):
    # make data list with only data from pixels in the defined box
    box_data = fits.getdata(filename)[:,970:1800,1070:1670]
    plt.title("Median Pixel Value Over Time: " + setting + " Mode", fontsize = 15)
    for i in range(box_data.shape[0]):
        # find the median of the pixels and plot for each image
        median_data = np.median(box_data[i,:,:])
        plt.scatter(i, median_data, c= color, label = filetype if i==0 else '')

if __name__ == "__main__":
    # fast plot
    plt.figure()
    plt.xlabel("Image #", fontsize = 10)
    plt.ylabel("Median Pixel Value")
    plot_avg(r"C:\Users\brie_\OneDrive - University of Arizona\Documents\2026 Summer Research LBT\Week_2\avg_fast_light_image_week2.fits", "Fast", "light", "blue")
    plot_avg(r"C:\Users\brie_\OneDrive - University of Arizona\Documents\2026 Summer Research LBT\Week_2\avg_fast_dark_image_week2.fits", "Fast", "dark", "red")
    plot_avg(r"C:\Users\brie_\OneDrive - University of Arizona\Documents\2026 Summer Research LBT\Week_2\net_fast_image_week2.fits", "Fast", "net", "black")
    plt.legend()
    plt.grid(True)
    plt.show()

    #medium plot
    plt.figure()
    plt.xlabel("Image #", fontsize = 10)
    plt.ylabel("Median Pixel Value")
    plot_avg(r"C:\Users\brie_\OneDrive - University of Arizona\Documents\2026 Summer Research LBT\Week_2\avg_medium_light_image_week2.fits", "Medium", "light", "blue")
    plot_avg(r"C:\Users\brie_\OneDrive - University of Arizona\Documents\2026 Summer Research LBT\Week_2\avg_medium_dark_image_week2.fits", "Medium", "dark", "red")
    plot_avg(r"C:\Users\brie_\OneDrive - University of Arizona\Documents\2026 Summer Research LBT\Week_2\net_medium_image_week2.fits", "Medium", "net", "black")
    plt.legend()
    plt.grid(True)
    plt.show()

    #slow plot
    plt.figure()
    plt.xlabel("Image #", fontsize = 10)
    plt.ylabel("Median Pixel Value")
    plot_avg(r"C:\Users\brie_\OneDrive - University of Arizona\Documents\2026 Summer Research LBT\Week_2\avg_slow_light_image_week2.fits", "Slow", "light", "blue")
    plot_avg(r"C:\Users\brie_\OneDrive - University of Arizona\Documents\2026 Summer Research LBT\Week_2\avg_slow_dark_image_week2.fits", "Slow", "dark", "red")
    plot_avg(r"C:\Users\brie_\OneDrive - University of Arizona\Documents\2026 Summer Research LBT\Week_2\net_slow_image_week2.fits", "Slow", "net", "black")
    plt.legend()
    plt.grid(True)
    plt.show()

## Figures Created
![Fast Scatterplot](Week_2/fast_mode_scatterplot.png)  
![Medium Scatterplot](Week_2/medium_mode_scatterplot.png)  
![Slow Scatterplot](Week_2/slow_mode_scatterplot.png)

### Notes
-in group meeting we talked about how median is better than mean for these cases, because it will not be widely affected by outliers but here the median and mean are approximately similar  
-it takes lots of memory to open multiple (here ~10) large fits files at once and my computer was able to handle it, but good to keep in mind to close unecessary applications while running the code  
-the scatterplots I got make sense, but the adjusted difference fit still doesn't appear to go through 0 is this what we should get? Will we artificially fit the linear line up slightly to account for this?

# Week 3

## Added Linear Fit and Made Linear Regime Cutoff
I modified last weeks code to create arrays of the x and y values to be plotted from a given .fits file.<br><br> 
Next I used np.corrcoef() to find the correlation coefficient of a set of the points in the scatterplot to find the linear regime, when I first did this I chose the first 12 points which for the fast mode difference produced r = 0.9995.<br><br>
Once this was plotted I used the y-value of the line at every x-value to find the absolute difference in the actual point and the line, then found whether the point was within 5% of the linear line.  
I found that the first points were often out of the 5% range since the values were small so any difference had a bigger impact, therefore I left them out of calculation of the linear regime limit.  
When I plotted my line I excluded the first point which increased the linearity, and only included the first 7 points as these were clearly in the linear regime.<br><br>
I plotted a linear line from the 2nd through 7th point for the light and difference plot of each mode, then found which points were within 5% of the linear line using my check_point_linearity function.  
Finally I created a horizontal line where the points were outside the 5% limit, aka outside of the linear regime. Any point at this line and above is outside the linear regime.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

# this function returns the median pixel value for each frame in a
# given .fits file in an x array and y array that can be plotted
# and manipulated
# filename is the path to the .fits file
def x_and_y(filename):
    # same as the plotting_avg_over_time.py, this limits the pixel
    # area to the box picked before 
    box_data = fits.getdata(filename)[:,970:1800,1070:1670]
    # creating lists to add the x and y values of points to
    x_values = []
    y_values = []
    # adding the median pixel value as y and the image as x
    # to create data points
    for i in range(box_data.shape[0]):
        # find the median of the pixels and plot for each image
        y_values += [np.median(box_data[i,:,:])]
        x_values += [i]
    # converting x-values to a numpy array so it can be used w/
    # numpy features later
    x_values = np.array(x_values)
    return x_values, y_values

# this function finds the correlation coefficient for a group of
# points, so I can find the linear regime points
# x and y are both arrays of x and y values
def linear_correlation_coef(x, y):
    # calculating the pearson r coefficient that tells you how linear
    # a group of points is from 0 (not correlated) to 1 (very linear)
    r = np.corrcoef(x, y)[0,1]
    return r

# this function takes the x and y values and the linear line constants
# previously determined, and finds which point first goes beyond
# 5% off the linear line
# m and b are both floats to make the linear line, x and y 
# are arrays of the points
def check_point_linearity(m, b, x, y):
    perfect_y = (m * x) + b
    abs_difference = np.abs(perfect_y - y)
    threshold = 0.05 * perfect_y
    for i in range(len(abs_difference)):
        if abs_difference[i] > threshold[i]:
            return perfect_y[i]
        
if __name__ == "__main__":

    #getting x and y values of all points (light file)
    x, y = x_and_y(r"C:\Users\brie_\OneDrive - University of Arizona\Documents\2026 Summer Research LBT\Week_2\avg_medium_light_image_week2.fits")

    #same but for difference file
    dx, dy = x_and_y(r"C:\Users\brie_\OneDrive - University of Arizona\Documents\2026 Summer Research LBT\Week_2\net_medium_image_week2.fits")

    # I used the first 7 points as this gave a high r value and
    # I wanted to be conservative
    lin_x = x[1:7]
    lin_y = y[1:7]
    lin_dx = dx[1:7]
    lin_dy = dy[1:7]

    #test linearity of these points
    print(linear_correlation_coef(lin_x, lin_y))
    print(linear_correlation_coef(lin_dx, lin_dy))

    # .polyfit creates constants needed to plot a line, the 1 tells
    # it to make a linear line aka slope and y-intercept
    m, b = np.polyfit(lin_x, lin_y, 1)
    dm, db = np.polyfit(lin_dx, lin_dy, 1)

    # finding point where linear regime ends, I excluded the
    # first two points since with small numbers the 5% cutoff
    # was too small and it placed the first points out of
    #the linear regime
    light_lin_regime_boundary = check_point_linearity(m, b, x[2:], y[2:])
    diff_lin_regime_boundary = check_point_linearity(dm, db, dx[2:], dy[2:])

    plt.figure()
    plt.title("Median Pixel Value Over Time: Medium Mode", fontsize = 15, pad = 20)
    plt.xlabel("Image #", fontsize = 10)
    plt.ylabel("Median Pixel Value")
    plt.scatter(x, y, c= "red", marker= '*', label = "light file values")
    plt.plot(x, m * x + b, 'r-', label = "light linear regime fit")
    plt.axhline(light_lin_regime_boundary, color='red', linestyle='--', linewidth=2, label = "linear regime boundary at " + str(int(light_lin_regime_boundary)))
    plt.scatter(dx, dy, c= "blue", marker= '*', label = "difference file values")
    plt.plot(dx, dm * dx + db, 'b-', label = "difference linear regime fit")
    plt.axhline(diff_lin_regime_boundary, color='blue', linestyle='--', linewidth=2, label = "linear regime boundary at " + str(int(diff_lin_regime_boundary)))
    plt.legend()
    plt.grid(True)
    plt.show()

## Figures Created
![Fast Linearity Analysis](Week_3\linfit_fast.png)
![Medium Linearity Analysis](Week_3\linfit_medium.png)
![Slow Linearity Analysis](Week_3\linfit_slow.png)

# Week 4

## Found Mean Dark and Mean Pseudo Flat For Distortion Reduction
This week we used new distortion files from the first part of the log, which simulated point sources with an illuminated grid on the detector. To reduce this data we first needed to find the mean dark file and mean pseudo flat file over 50 files each.<br><br>
The dark files were just the dark reading and had two images, a baseline and fully completed scan. We used the fully completed scan to make the mean file, so there was only one image in the mean dark file.<br><br>
For the pseudo flat files, we found the mean the same way as the dark with the second, fully illuminated image in each file. Next we subtracted the mean dark file from the mean pseudo flat, and normalized the file by the median method: normalized = x/median(x). 

## Reduced Distortion Data Using Reduction Equation
Next we reduced each of the distortion/object fits files (the ones with the grid of light) using the reduction formula:<br><br>
reduction = (object-(mean dark))/(normalized, dark subtracted pseudo flat)<br><br>
I applied this formula to every distortion fits file using our calculated dark and pseudo flat files, then saved these sequentially

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob

# creates a list of the .fits files within a folder, the path to 
# the folder with\*.fits is the file_path
def create_file_list(file_paths):
    file_list = glob.glob(file_paths)
    return file_list

# averages the data from a list of .fits files, filetype is a string
# that identifies the type of file in the header
def average_files(filelist, filetype):

    #getting the second/non baseline image from each file
    file_concat = [fits.getdata(file)[1] for file in filelist]

    #finding the mean (averaging the files)
    #.astype(np.float32) shrinks the acuracy of the estimation, since the mean auto
    #defaults to float64, so this will make the file smaller
    final_file_data = np.mean(file_concat, axis=0).astype(np.float32)

    #carry over first image header and add information on averaging
    original_header = fits.getheader(filelist[0])
    new_header = original_header.copy()
    # will return something of the style: averaged 10 light files
    new_header['HISTORY'] = f"averaged {len(filelist)} {filetype} files"
    new_header['METHOD'] = 'MEAN'

    return final_file_data, new_header

# creates a file with a given filename string (what the new file will 
# be named) the data that will be in the file and the new header
def create_file(filename, filedata, header):
    hdu = fits.PrimaryHDU(filedata, header=header)
    hdu.writeto(filename, overwrite=True)

if __name__ == "__main__":
    #get filelists 
    dark_files = create_file_list(r"C:\Users\brie_\OneDrive - University of Arizona\Documents\2026 Summer Research LBT\Downloaded Files\Week 4\darks\*.fits.gz")
    object_files = create_file_list(r"C:\Users\brie_\OneDrive - University of Arizona\Documents\2026 Summer Research LBT\Downloaded Files\Week 4\observations\*.fits.gz")
    pseudo_flat_files = create_file_list(r"C:\Users\brie_\OneDrive - University of Arizona\Documents\2026 Summer Research LBT\Downloaded Files\Week 4\pseudo_flats\*.fits.gz")

    #get mean of dark files
    mean_dark = average_files(dark_files, "dark")

    #get mean of pseudo flat files
    mean_pseudo_flat = average_files(pseudo_flat_files, "pseudo flat")

    # subtracting dark mean and normalizing the pseudo flat files.
    # adjusting header accordingly
    difference = mean_pseudo_flat[0] - mean_dark[0]
    normalized_pflat = difference / np.median(difference)
    norm_header = fits.getheader(pseudo_flat_files[0]).copy()
    norm_header['HISTORY'] = 'subtracted mean of dark files from mean of pseudo flat files, then normalized dark removed pseudo flats by dividing by the median'
    
    #applying reduction equation to object/distortion files
    object_data = [fits.getdata(file)[1] for file in object_files]
    reduced_data = [(obj-mean_dark[0])/normalized_pflat for obj in object_data]
    reduced_header = fits.getheader(object_files[0]).copy()
    reduced_header['HISTORY'] = 'reduced the distortion data by subtracting the mean dark and dividing by the normalized mean pseudoflat'
    
    '''
    # creating new .fits files:

    #mean dark file
    create_file("distortion_mean_dark.fits", mean_dark[0], mean_dark[1])

    # normalized pseudo flat file
    create_file("normalized_mean_pseudo_flats.fits", normalized_pflat, norm_header)

    # reduced data files
    for i, img in enumerate(reduced_data):
        create_file(f"reduced_distortion_{i}.fits", img , reduced_header)
    '''

## Checked New .fits Files 
After creating the mean dark, normalized pseudo file, and reduced distortion files I plotted them to make sure they looked normal and note any things I was curious or confused about.  
When plotting the final reduced distortion file, the range was originally way too large and I couldn't see the dots anymore, so I reduced the range of the pixel values to see the differences clearer. I also plotted the reduced file with a range similar to the original distortion file to compare the two. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob


if __name__ == "__main__":
    # looking at distortion file
    obj_file = r"C:\Users\brie_\OneDrive - University of Arizona\Documents\2026 Summer Research LBT\Downloaded Files\Week 4\observations\lm_260429_000001.fits.gz"
    plt.figure()
    plt.title("Distortion File", fontsize = 15, pad = 20)
    plt.xlabel("x pixels", fontsize = 10)
    plt.ylabel("y pixels")
    plt.imshow(fits.getdata(obj_file)[1], cmap="plasma")
    plt.colorbar()

    # showing mean dark file (only used the 2nd frame in 
    # creating it so only one image)
    distortion_mean_dark = r"C:\Users\brie_\OneDrive - University of Arizona\Documents\2026 Summer Research LBT\Week_4\distortion_mean_dark.fits"
    plt.figure()
    plt.title("Mean Dark File, Second Frame", fontsize = 15, pad = 20)
    plt.xlabel("x pixels", fontsize = 10)
    plt.ylabel("y pixels")
    plt.imshow(fits.getdata(distortion_mean_dark), cmap="plasma")
    plt.colorbar()

    # showing normalized, dark subtracted mean pseudo flat
    normalized_mean_pseudo_flats = r"C:\Users\brie_\OneDrive - University of Arizona\Documents\2026 Summer Research LBT\Week_4\normalized_mean_pseudo_flats.fits"
    plt.figure()
    plt.title("Normalized, Dark Subtracted, Mean Pseudo File", fontsize = 15, pad = 20)
    plt.xlabel("x pixels", fontsize = 10)
    plt.ylabel("y pixels")
    plt.imshow(fits.getdata(normalized_mean_pseudo_flats), cmap="plasma")
    plt.colorbar()

    # showing reduced distortion file 
    reduced = r"C:\Users\brie_\OneDrive - University of Arizona\Documents\2026 Summer Research LBT\Week_4\reduced_distortion_data\reduced_distortion_0.fits"
    plt.figure()
    plt.title("Reduced Distortion File, Small Range", fontsize = 15, pad = 20)
    plt.xlabel("x pixels", fontsize = 10)
    plt.ylabel("y pixels")
    # I decreased the range to show the differences in the dots more clearly
    plt.imshow(fits.getdata(reduced), cmap="plasma", vmin = 0, vmax = 1000)
    plt.colorbar()
    plt.figure()
    # here I attempted to make the range what it was in the original distortion file
    plt.title("Reduced Distortion File, Original Range", fontsize = 15, pad = 20)
    plt.xlabel("x pixels", fontsize = 10)
    plt.ylabel("y pixels")
    plt.imshow(fits.getdata(reduced), cmap="plasma", vmin = 0, vmax = 4100)
    plt.colorbar()
    



## Figures Created
![Mean Dark File](Week_4\mean_dark_distortion_plotted.png)
![Normalized Mean Pseudo Flat](Week_4\norm_mean_pseudoflat_plotted.png)
![Original Distortion File](Week_4\distortion_plotted.png)
![Reduced Distortion File, Original Range](Week_4\reduced_distortion_og_range_plotted.png)
![Reduced Distortion File, Small Range](Week_4\reduced_distortion_small_range_plotted.png)